In [1]:
import os
import re
import random
import unicodedata
import difflib

import pandas as pd
import numpy as np

BASE_DIR = ".."

TRAIN_DIR = os.path.join(BASE_DIR, "data", "train")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

GROUND_TRUTH = os.path.join(TRAIN_DIR, "train_ground_truth.tsv")
SOURCE1 = os.path.join(TRAIN_DIR, "train_source1.tsv")
SOURCE2 = os.path.join(TRAIN_DIR, "train_source2.tsv")
SOURCE3 = os.path.join(TRAIN_DIR, "train_source3.tsv")

CHUNK_SIZE = 100_000
SAMPLE_SIZE_PER_SOURCE = 3000
RANDOM_SEED = 42

print("Ground truth:", GROUND_TRUTH)
print("Source 1:", SOURCE1)
print("Source 2:", SOURCE2)
print("Source 3:", SOURCE3)

Ground truth: ../data/train/train_ground_truth.tsv
Source 1: ../data/train/train_source1.tsv
Source 2: ../data/train/train_source2.tsv
Source 3: ../data/train/train_source3.tsv


In [2]:
rng = random.Random(RANDOM_SEED)

s2_sample = []
s3_sample = []

seen_s2 = 0
seen_s3 = 0

for chunk in pd.read_csv(
    GROUND_TRUTH,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):
    for _, row in chunk.iterrows():

        s1_id = row["source1_entity_id"]
        matched = row["matched_entity_ids"]

        if not matched:
            continue

        # Extract S2/S3 IDs without loading all relationships.
        target_ids = re.findall(r"S[23]-\d+", matched)

        match_count = len(target_ids)

        for target_id in target_ids:

            if target_id.startswith("S2-"):
                seen_s2 += 1

                record = {
                    "source1_entity_id": s1_id,
                    "target_entity_id": target_id,
                    "target_source": "S2",
                    "match_count_for_s1": match_count
                }

                if len(s2_sample) < SAMPLE_SIZE_PER_SOURCE:
                    s2_sample.append(record)
                else:
                    j = rng.randint(1, seen_s2)

                    if j <= SAMPLE_SIZE_PER_SOURCE:
                        s2_sample[j - 1] = record

            elif target_id.startswith("S3-"):
                seen_s3 += 1

                record = {
                    "source1_entity_id": s1_id,
                    "target_entity_id": target_id,
                    "target_source": "S3",
                    "match_count_for_s1": match_count
                }

                if len(s3_sample) < SAMPLE_SIZE_PER_SOURCE:
                    s3_sample.append(record)
                else:
                    j = rng.randint(1, seen_s3)

                    if j <= SAMPLE_SIZE_PER_SOURCE:
                        s3_sample[j - 1] = record

print("S2 relationships seen:", seen_s2)
print("S3 relationships seen:", seen_s3)

print("Sampled S2 pairs:", len(s2_sample))
print("Sampled S3 pairs:", len(s3_sample))

S2 relationships seen: 3693619
S3 relationships seen: 3944746
Sampled S2 pairs: 3000
Sampled S3 pairs: 3000


In [3]:
pair_sample = pd.DataFrame(
    s2_sample + s3_sample
)

pair_sample = pair_sample.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

print("Total sampled positive pairs:", len(pair_sample))

pair_sample.head(10)

Total sampled positive pairs: 6000


,source1_entity_id,target_entity_id,target_source,match_count_for_s1
0,S1-971243218,S2-671137794,S2,4
1,S1-332426239,S3-898011295,S3,3
2,S1-833610609,S2-487526686,S2,4
3,S1-491832136,S2-41491591,S2,3
4,S1-830862319,S3-38449752,S3,6
5,S1-120719938,S2-349307671,S2,4
6,S1-612106862,S2-535298092,S2,5
7,S1-158715097,S2-334841182,S2,5
8,S1-540720169,S2-581122477,S2,5
9,S1-315928141,S2-802992830,S2,6


In [4]:
sample_s1_ids = set(
    pair_sample["source1_entity_id"]
)

source1_sample_parts = []

for chunk in pd.read_csv(
    SOURCE1,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):
    selected = chunk[
        chunk["entity_id"].isin(sample_s1_ids)
    ]

    if len(selected) > 0:
        source1_sample_parts.append(selected)

source1_sample = pd.concat(
    source1_sample_parts,
    ignore_index=True
)

source1_sample = source1_sample.rename(
    columns={
        "entity_id": "source1_entity_id",
        "business_name": "s1_business_name",
        "business_address": "s1_business_address",
        "country": "s1_country"
    }
)

print("Retrieved S1 records:", len(source1_sample))

Retrieved S1 records: 5992


In [5]:
sample_s2_ids = set(
    pair_sample.loc[
        pair_sample["target_source"] == "S2",
        "target_entity_id"
    ]
)

sample_s3_ids = set(
    pair_sample.loc[
        pair_sample["target_source"] == "S3",
        "target_entity_id"
    ]
)

source2_sample_parts = []

for chunk in pd.read_csv(
    SOURCE2,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):
    selected = chunk[
        chunk["entity_id"].isin(sample_s2_ids)
    ]

    if len(selected) > 0:
        source2_sample_parts.append(selected)


source3_sample_parts = []

for chunk in pd.read_csv(
    SOURCE3,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):
    selected = chunk[
        chunk["entity_id"].isin(sample_s3_ids)
    ]

    if len(selected) > 0:
        source3_sample_parts.append(selected)


source2_sample = pd.concat(
    source2_sample_parts,
    ignore_index=True
)

source3_sample = pd.concat(
    source3_sample_parts,
    ignore_index=True
)

print("Retrieved S2 records:", len(source2_sample))
print("Retrieved S3 records:", len(source3_sample))

Retrieved S2 records: 3000
Retrieved S3 records: 3000


In [6]:
target_sample = pd.concat(
    [
        source2_sample.assign(target_source="S2"),
        source3_sample.assign(target_source="S3")
    ],
    ignore_index=True
)

target_sample = target_sample.rename(
    columns={
        "entity_id": "target_entity_id",
        "business_name": "target_business_name",
        "business_address": "target_business_address",
        "country": "target_country"
    }
)

pair_data = (
    pair_sample
    .merge(
        source1_sample,
        on="source1_entity_id",
        how="left"
    )
    .merge(
        target_sample[
            [
                "target_entity_id",
                "target_source",
                "target_business_name",
                "target_business_address",
                "target_country"
            ]
        ],
        on=["target_entity_id", "target_source"],
        how="left"
    )
)

print("Final pair rows:", len(pair_data))
pair_data.head()

Final pair rows: 6000


,source1_entity_id,target_entity_id,target_source,match_count_for_s1,s1_business_name,s1_business_address,s1_country,target_business_name,target_business_address,target_country
0,S1-971243218,S2-671137794,S2,4,Unified Asset Worldwide,"7530 Locust Avenue, Gary, IN",US,UNIFIED ASSET,"7530 LOCUST AVENUE, GARY, IN",US
1,S1-332426239,S3-898011295,S3,3,Filea Corporate LLC,"2011 Brown Street, City Of Wisconsin Rapids, WI",US,Filea Corporate Llc,"2011 Brown St, City Of Wisconsin Rapids, Wisco...",US
2,S1-833610609,S2-487526686,S2,4,Direct Continental Rothschild LLC,"OH, Youngstown, 2821 Hamman Drive",US,Direct Continental Róthschild Llc,"2821 HAMMAN DRIVE, YOUNGSTOWN, OH",US
3,S1-491832136,S2-41491591,S2,3,Hager Shore LLC,"47 Overton Road, Roanoke Rapids, NC",US,Hager Shore,,US
4,S1-830862319,S3-38449752,S3,6,Mitchell & Candelaria Strategies,"Mead, WA, 13511 Bruce Road",US,mcandelaria.com,"Mead, Washington, Bruce Road",US


In [7]:
print(
    "Missing S1 names:",
    pair_data["s1_business_name"].isna().sum()
)

print(
    "Missing target names:",
    pair_data["target_business_name"].isna().sum()
)

print(
    "Missing S1 addresses:",
    pair_data["s1_business_address"].isna().sum()
)

print(
    "Missing target addresses:",
    pair_data["target_business_address"].isna().sum()
)

Missing S1 names: 0
Missing target names: 0
Missing S1 addresses: 0
Missing target addresses: 0


In [8]:
def normalize_text(value):
    value = "" if value is None else str(value)

    value = unicodedata.normalize(
        "NFKC",
        value
    )

    value = value.casefold()

    value = re.sub(
        r"[^\w\s]",
        " ",
        value,
        flags=re.UNICODE
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
        flags=re.UNICODE
    )

    return value.strip()

In [9]:
def token_jaccard(a, b):
    a_tokens = set(
        re.findall(r"\w+", normalize_text(a))
    )

    b_tokens = set(
        re.findall(r"\w+", normalize_text(b))
    )

    if not a_tokens and not b_tokens:
        return 1.0

    if not a_tokens or not b_tokens:
        return 0.0

    intersection = len(a_tokens & b_tokens)
    union = len(a_tokens | b_tokens)

    return intersection / union


def sequence_similarity(a, b):
    a = normalize_text(a)
    b = normalize_text(b)

    if not a and not b:
        return 1.0

    if not a or not b:
        return 0.0

    return difflib.SequenceMatcher(
        None,
        a,
        b
    ).ratio()

In [10]:
pair_data["name_exact_raw"] = (
    pair_data["s1_business_name"]
    == pair_data["target_business_name"]
)

pair_data["address_exact_raw"] = (
    pair_data["s1_business_address"]
    == pair_data["target_business_address"]
)

pair_data["name_normalized_exact"] = (
    pair_data["s1_business_name"].map(normalize_text)
    ==
    pair_data["target_business_name"].map(normalize_text)
)

pair_data["address_normalized_exact"] = (
    pair_data["s1_business_address"].map(normalize_text)
    ==
    pair_data["target_business_address"].map(normalize_text)
)

pair_data["name_jaccard"] = [
    token_jaccard(a, b)
    for a, b in zip(
        pair_data["s1_business_name"],
        pair_data["target_business_name"]
    )
]

pair_data["address_jaccard"] = [
    token_jaccard(a, b)
    for a, b in zip(
        pair_data["s1_business_address"],
        pair_data["target_business_address"]
    )
]

pair_data["name_sequence_similarity"] = [
    sequence_similarity(a, b)
    for a, b in zip(
        pair_data["s1_business_name"],
        pair_data["target_business_name"]
    )
]

pair_data["address_sequence_similarity"] = [
    sequence_similarity(a, b)
    for a, b in zip(
        pair_data["s1_business_address"],
        pair_data["target_business_address"]
    )
]

pair_data["country_same"] = (
    pair_data["s1_country"]
    == pair_data["target_country"]
)

In [11]:
summary = pd.DataFrame({
    "metric": [
        "Name raw exact",
        "Name normalized exact",
        "Address raw exact",
        "Address normalized exact",
        "Country same",
        "Target address missing"
    ],
    "count": [
        pair_data["name_exact_raw"].sum(),
        pair_data["name_normalized_exact"].sum(),
        pair_data["address_exact_raw"].sum(),
        pair_data["address_normalized_exact"].sum(),
        pair_data["country_same"].sum(),
        (
            pair_data["target_business_address"]
            .str.strip()
            == ""
        ).sum()
    ]
})

summary["percentage"] = (
    summary["count"]
    / len(pair_data)
    * 100
)

summary

,metric,count,percentage
0,Name raw exact,268,4.466667
1,Name normalized exact,1329,22.150000
2,Address raw exact,144,2.400000
3,Address normalized exact,531,8.850000
4,Country same,6000,100.000000
5,Target address missing,258,4.300000


In [12]:
similarity_summary = pair_data[
    [
        "name_jaccard",
        "name_sequence_similarity",
        "address_jaccard",
        "address_sequence_similarity"
    ]
].describe(
    percentiles=[
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

similarity_summary

,name_jaccard,name_sequence_similarity,address_jaccard,address_sequence_similarity
count,6000.000000,6000.000000,6000.000000,6000.000000
mean,0.616026,0.792528,0.599709,0.750285
std,0.332090,0.248881,0.266278,0.237308
min,0.000000,0.000000,0.000000,0.000000
1%,0.000000,0.083333,0.000000,0.000000
5%,0.000000,0.109589,0.124632,0.279658
10%,0.000000,0.454545,0.250000,0.444444
25%,0.500000,0.727273,0.428571,0.643473
50%,0.666667,0.878049,0.625000,0.823529
75%,1.000000,0.962963,0.800000,0.920000


In [13]:
source_similarity_summary = (
    pair_data
    .groupby("target_source")[
        [
            "name_jaccard",
            "name_sequence_similarity",
            "address_jaccard",
            "address_sequence_similarity"
        ]
    ]
    .agg(
        [
            "mean",
            "median",
            "min",
            "max"
        ]
    )
)

source_similarity_summary

name_jaccard                     name_sequence_similarity  \
                      mean    median  min  max                     mean   
target_source                                                             
S2                0.612035  0.666667  0.0  1.0                 0.786004   
S3                0.620016  0.666667  0.0  1.0                 0.799052   

                                address_jaccard                      \
              median   min  max            mean    median  min  max   
target_source                                                         
S2             0.880  0.00  1.0        0.679850  0.705882  0.0  1.0   
S3             0.875  0.05  1.0        0.519568  0.500000  0.0  1.0   

              address_sequence_similarity                      
                                     mean    median  min  max  
target_source                                                  
S2                               0.790698  0.862745  0.0  1.0  
S3                               0.709871  0.785714  0.0  1.0

In [14]:
match_count_analysis = (
    pair_data
    .groupby(
        [
            "target_source",
            "match_count_for_s1"
        ]
    )[
        [
            "name_jaccard",
            "name_sequence_similarity",
            "address_jaccard",
            "address_sequence_similarity"
        ]
    ]
    .mean()
)

match_count_analysis

name_jaccard  name_sequence_similarity  \
target_source match_count_for_s1                                           
S2            1                       0.678028                  0.848927   
              2                       0.615006                  0.800219   
              3                       0.603711                  0.774650   
              4                       0.593222                  0.774030   
              5                       0.613378                  0.782408   
              6                       0.638835                  0.798674   
              7                       0.634767                  0.813080   
              8                       0.585367                  0.772718   
              9                       0.662222                  0.829224   
              10                      0.500000                  0.950588   
S3            1                       0.685301                  0.839332   
              2                       0.632439                  0.819030   
              3                       0.617055                  0.802337   
              4                       0.616160                  0.800362   
              5                       0.631112                  0.799361   
              6                       0.608162                  0.780600   
              7                       0.576869                  0.784342   
              8                       0.643462                  0.802498   
              9                       0.666667                  0.736423   
              10                      0.773333                  0.881660   
              11                      0.500000                  0.581818   

                                  address_jaccard  address_sequence_similarity  
target_source match_count_for_s1                                                
S2            1                          0.668086                     0.791648  
              2                          0.689798                     0.803188  
              3                          0.668065                     0.789481  
              4                          0.675696                     0.791181  
              5                          0.684863                     0.789097  
              6                          0.702027                     0.799964  
              7                          0.651619                     0.764483  
              8                          0.713573                     0.769968  
              9                          0.746693                     0.809515  
              10                         0.708333                     0.968629  
S3            1                          0.479022                     0.717480  
              2                          0.504880                     0.708763  
              3                          0.530728                     0.712317  
              4                          0.525241                     0.718071  
              5                          0.512918                     0.703797  
              6                          0.515809                     0.699905  
              7                          0.534927                     0.715197  
              8                          0.484553                     0.692367  
              9                          0.544022                     0.778618  
              10                         0.385000                     0.639727  
              11                         1.000000                     0.657143

In [15]:
pair_data["combined_similarity"] = (
    0.5 * pair_data["name_sequence_similarity"]
    +
    0.5 * pair_data["address_sequence_similarity"]
)

hardest_pairs = (
    pair_data
    .sort_values("combined_similarity")
    [
        [
            "source1_entity_id",
            "target_entity_id",
            "target_source",
            "match_count_for_s1",
            "s1_business_name",
            "target_business_name",
            "s1_business_address",
            "target_business_address",
            "s1_country",
            "target_country",
            "name_sequence_similarity",
            "address_sequence_similarity",
            "combined_similarity"
        ]
    ]
    .head(30)
)

hardest_pairs

,source1_entity_id,target_entity_id,target_source,match_count_for_s1,s1_business_name,target_business_name,s1_business_address,target_business_address,s1_country,target_country,name_sequence_similarity,address_sequence_similarity,combined_similarity
4480,S1-617953347,S2-280282680,S2,6,Tirupati Systems Private Limited,तिरुपति सिस्टम्स प्राइवेट लिमिटेड,"A8, Green Acre Housing Society Bibwewadi, Mark...","A8, PUNE, महाराष्ट्र",India,India,0.093750,0.186047,0.139898
4424,S1-256063388,S3-753087395,S3,4,Aditya Infrastructure Limited,आदित्य इंफ्रास्ट्रक्चर लिमिटेड,"Cyb-1, Stpi Incubation, Software Technology Pa...","Door No 764 Cyb-1, Jodhpur, राजस्थान",India,India,0.067797,0.225806,0.146802
131,S1-667386889,S3-915549335,S3,4,Vijay Solutions,विजय सॉल्यूशंस,"C/O Baburam, Jwahar Chak, Mohalla And Post Nau...","Gorakhpur, UP, C/o Baburam",India,India,0.068966,0.231579,0.150272
3384,S1-577155480,S2-476829495,S2,5,Tirupati Technology Private Limited,తిరుపతి టెక్నాలజీ ప్రైవేట్ లిమిటెడ్,"# Door No.5-336, Mancherial, Adilabad, Telanga...","Andhra Pradesh, ADILABAD, # DOOR NO.5-336",India,India,0.090909,0.236364,0.163636
3742,S1-858944037,S3-121363377,S3,5,Balaji Foundation Private Limited,ಬಾಲಾಜಿ ಫೌಂಡೇಶನ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,"# 67, Pranav 25Th Cross Jayanagar, 6Th Block, ...","Plt 531 F2, Urban, KA",India,India,0.098361,0.239130,0.168746
16,S1-61869418,S2-496027602,S2,5,Real Services LLP,रियल सर्विसेज एलएलपी,"Maharashtra, E-107, Pune, Kailas Laxman, Neher...","E-107, PUNE, BARAMATI, Maharashtra",India,India,0.111111,0.226804,0.168958
476,S1-183522604,S3-561335693,S3,7,Surya Solutions,सूर्य सॉल्यूशंस,"A-322, Floor - 3, 1, Antop Hill Warehousing, V...","A-322, Mumbai City, MH, Mumbai",India,India,0.066667,0.300752,0.183709
608,S1-284776107,S3-637526828,S3,4,Vision Developers Private Limited,విజన్ డెవలపర్స్ ప్రైవేట్ లిమిటెడ్,"Plot No. 201, Ram Residence, Beeramguda, New S...","201, Medak, TG",India,India,0.096774,0.272727,0.184751
2366,S1-339463448,S2-932878592,S2,5,Shiv Infotech Private Limited,ಶಿವ್ ಇನ್‌ಫೋಟೆಕ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,"3D, Uttam Sagar Appts, 176/A S V Road, Yadagir...","ಕರ್ನಾಟಕ, null, MYSURU, MYSORE, #3D",India,India,0.105263,0.288889,0.197076
3604,S1-304235697,S2-762838324,S2,6,Surya Investment Private Limited,सूर्या इन्वेस्टमेंट प्राइवेट लिमिटेड,"Shop No. F/4, Shivdarshan Appartment No. 2 Dig...","NAGPUR, BLOCK E-859 SHOP NO. F/4, NAGPUR, महार...",India,India,0.090909,0.304000,0.197455


In [16]:
easiest_pairs = (
    pair_data
    .sort_values(
        "combined_similarity",
        ascending=False
    )
    [
        [
            "source1_entity_id",
            "target_entity_id",
            "target_source",
            "match_count_for_s1",
            "s1_business_name",
            "target_business_name",
            "s1_business_address",
            "target_business_address",
            "name_sequence_similarity",
            "address_sequence_similarity"
        ]
    ]
    .head(20)
)

easiest_pairs

,source1_entity_id,target_entity_id,target_source,match_count_for_s1,s1_business_name,target_business_name,s1_business_address,target_business_address,name_sequence_similarity,address_sequence_similarity
3236,S1-251698975,S2-859912062,S2,2,Human Rights Network LLC,Human Rights Network Llc,"2833 Beacon Avenue, Spokane, WA","2833 Beacon Avenue, SPOKANE, WA",1.0,1.0
335,S1-402437438,S2-980971177,S2,3,Nrj Mediatech Private Limited,Nrj Mediatech [Private-Limited],"C/O Rajpati, Booth No 47 Sector-2 Part, Rohtak...","C/O RAJPATI, BOOTH NO 47 SECTOR-2 PART, ROHTAK...",1.0,1.0
5888,S1-578805165,S2-172038304,S2,4,"Gibson & Sanchez Arts, LLC","GIBSON & SANCHEZ ARTS, ([LLC])","748 Twp Rd 109, South Point, OH","748 TWP RD 109, SOUTH POINT, OH",1.0,1.0
284,S1-699433275,S2-68622563,S2,6,Mendoza Clear Chemical,Mendoza Clear Chemical,"74 Catfish Lane, Scranton, AR","74 CATFISH LANE, SCRANTON, AR",1.0,1.0
1981,S1-774830436,S2-527109961,S2,5,Mckenzie and Marquez LLC,Mckenzie and Marquez LLC,"36 Hay Field Drive, Gorham, ME","36 HAY FIELD DRIVE, GORHAM, ME",1.0,1.0
2680,S1-547256104,S2-539012565,S2,6,Moore Quality Consulting LLC,MOORE QUALITY CONSULTING LLC,"2550 Elmhurst Avenue, Beaverton, OR","2550 Elmhurst Avenue, BEAVERTON, OR",1.0,1.0
2695,S1-128874388,S2-333622861,S2,4,Tilak & Sons Limited,Tilak & Sons Limited,"Flat#201, Eternal Pride, Plot 94/D, Hig Huda L...","FLAT#201, ETERNAL PRIDE, PLOT 94/D, HIG HUDA L...",1.0,1.0
5480,S1-642259385,S2-484900340,S2,3,F/P Photonics,F/P Photonics,"9615 Cooper Church Dr, Louisville, KY","9615 COOPER CHURCH DR, LOUISVILLE, KY",1.0,1.0
5471,S1-243735346,S2-929709025,S2,4,Dental Group,DENTAL GROUP,"475 Jay Street, Rochester, NY","475 JAY STREET, ROCHESTER, NY",1.0,1.0
5461,S1-393819961,S2-237064844,S2,3,Onyx Fuse LLC,ONYX FUSE LLC,"8929 Hiwassee Street, Charleston, TN","8929 HIWASSEE STREET, CHARLESTON, TN",1.0,1.0


In [17]:
pair_data.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "pair_eda_sample.csv"
    ),
    index=False
)

summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "pair_eda_summary.csv"
    ),
    index=False
)

print("Saved pair EDA outputs.")

Saved pair EDA outputs.
